# Validation of predicted spectral gaps in the generated instances

$\renewcommand{\ket}[1]{\left|#1\right\rangle}\renewcommand{\bra}[1]{\left\langle #1\right|}\renewcommand{\braket}[2]{\left\langle #1 \middle| #2 \right\rangle}\renewcommand{\ketbra}[2]{\left|#1\right\rangle\!\left\langle #2\right|}$
This notebook shows the agreement between the spectral gaps predicted by our fitting rule and the actual ones associated with our instances and the conservative schedule. 

## Import

In [1]:
import sys

if ".." not in sys.path:
    sys.path.append("..")

from functools import cache
import math
import numpy as np
from pathlib import Path
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
import sympy as sp

## Loading the schedules and spectral gaps

The folder `estimation_variance/` contains newly generated SK instances for $n = 5, ..., 30$ for which, in the subfolder `estimation_variance/schedule_outputs_conservative/`, we have generated a conservative schedule of inverse temperatures $0 = \beta_0 < \ldots < \beta_L = 16$ satisfying the desired overlap rule between Gibbs states at consecutive temperatures. 

We also want to make sure that the fitted spectral gap rule $\delta(\beta, n)$ gives results that are reasonably close to the actual spectral gap associated with the inverse temperature of the schedule. A large mismatch may imply that the spectral filter over- or under-estimate the degree necessary to keep the stationary component only, silently introducing an error larger than the one budgetted for. 

The code to run this analysis, for $n = 5, ..., 10$ only, is in `estimation_variance/validate_schedule_gaps.{py,slurm}`. To get the fitted spectral gap one can use the module `monaqa2.data.spectral_gap.py`:

In [2]:
from monaqa2.data.spectral_gap import get_spectral_gap_fit_by_n

beta = 0.5
n = 5

A, b = get_spectral_gap_fit_by_n(
    'uniform', np.inf, beta,
    statistic="mean+std", n_min=5, n_max=10,
)
delta_fit = A * np.exp(-max(b, 0.0) * n)

Note that this method uses data with $\beta$ belonging to a finite grid. Therefore, intermediate temperatures are obtained by interpolating $\log A$ and $b$.

The following code loads the actual pre-calculated spectral gaps. 

In [3]:
GAP_DIRECTORY = Path("../estimation_variance/schedule_outputs_conservative")
EPS, GAP_RESOLUTION = 1e-2, 1e-12

# Load the pre-calculated spectral gaps, see them in column 'gap_actual'
schedule_gaps = pd.concat([
    pd.read_csv(path, sep=r"\s+", comment="#",
                usecols=["n", "idx", "proposal", "beta", "gap_actual"])
    for n in range(5, 11) for idx in range(100)
    if (path := GAP_DIRECTORY / f"gaps_{n}_idx{idx}.txt").is_file()
], ignore_index=True)

# Add the length of the schedule
schedule_gaps["L"] = schedule_gaps.groupby(["n", "idx"])["beta"].transform("nunique")

Using the method above we add to this table the fitted spectral gaps.

In [4]:
@cache
def fitted_gap(n, proposal, beta):
    A, b = get_spectral_gap_fit_by_n(
        proposal, np.inf, beta, statistic="mean+std", n_min=5, n_max=10)
    return A * np.exp(-max(b, 0.0) * n)

schedule_gaps["gap_fit"] = [
    fitted_gap(row.n, row.proposal, row.beta)
    for row in schedule_gaps.itertuples()
]

## Validation of the spectral gaps

We compare the numerical spectral gaps with the fitted predictions for the uniform and Hamiltonian-simulation proposals, at each system size $n=5,\ldots,10$. The inverse temperatures along the conservative schedule are grouped into $(0,1]$, $(1,2]$, $(2,4]$, and $(4,16]$.

For each group, the **expected gap** is evaluated using `fitted_gap(n, proposal, beta)` at the **upper endpoint** of the temperature interval ($\beta=1,2,4,16$). The median, 10th percentile, and minimum are computed from all numerical gaps observed across the 100 instances and all schedule steps falling in that interval. The quantiles therefore describe the **pooled step-wise gaps**, not the distribution of one minimum per instance. Unlike the fitted value, they do not refer to one fixed temperature.

In [5]:
d = schedule_gaps.copy()
d["beta_range"] = pd.cut(d["beta"], [0, 1, 2, 4, 16])

summary = d.groupby(["proposal", "n", "beta_range"], observed=True)["gap_actual"].agg(
    **{
        "Median gap": "median",
        "10-percentile": lambda x: x.quantile(0.1),
        "Minimum gap": "min",
    }
)

summary.insert(0, "Expected gap", [
    fitted_gap(n, proposal, beta_range.right)
    for proposal, n, beta_range in summary.index
])

print(summary.to_string(float_format=lambda x: f"{x:.3e}"))

                        Expected gap  Median gap  10-percentile  Minimum gap
proposal n  beta_range                                                      
layden   5  (0, 1]         1.229e-01   1.366e-01      6.738e-02    2.580e-02
            (1, 2]         1.084e-01   1.061e-01      5.627e-02    2.562e-02
            (2, 4]         1.063e-01   1.053e-01      5.709e-02    2.600e-02
            (4, 16]        1.044e-01   1.008e-01      5.737e-02    2.494e-02
         6  (0, 1]         1.038e-01   1.145e-01      5.399e-02    2.778e-02
            (1, 2]         8.916e-02   8.334e-02      4.029e-02    2.776e-02
            (2, 4]         8.649e-02   7.925e-02      3.900e-02    2.802e-02
            (4, 16]        8.418e-02   7.949e-02      3.877e-02    2.273e-02
         7  (0, 1]         8.761e-02   1.169e-01      6.069e-02    2.377e-02
            (1, 2]         7.330e-02   8.267e-02      4.256e-02    2.239e-02
            (2, 4]         7.040e-02   7.700e-02      3.759e-02    1.870e-02

The fitted gaps are generally comparable to the observed medians, but they are not instance-wise lower bounds. 

## Spectral filter on correct spectral gaps

For a spectral gap $\delta\in(0,1)$, our even Chebyshev filter is
$$
p_{d,\delta}(x)=\frac{T_d(x/(1-\delta))}{T_d(1/(1-\delta))},
$$
for $d$ even. It preserves $p_{d,\delta}(1)=1$ and, writing the phase gap as $\theta=\arccos(1-\delta)$, satisfies
$$
\max_{|x|\leq1-\delta}|p_{d,\delta}(x)|
\leq\frac{1}{\cosh\!\left[d\,\operatorname{arcosh}(1/(1-\delta))\right]}
\leq 2e^{-d\theta}.
$$
Therefore, to ensure a filter error at most $\varepsilon$, it is sufficient to choose an even degree satisfying
$$
d\geq\frac{\ln(2/\varepsilon)}{\theta}
=\frac{\ln(2/\varepsilon)}{\arccos(1-\delta)}.
$$

In [6]:
from scipy.special import eval_chebyt

delta, eps = 0.01, 1e-2

def degree(delta, eps):
    return 2 * int(np.ceil(np.log(2 / eps) / (2 * np.arccos(1 - delta))))

def filter(x, d):
    return eval_chebyt(d, x / (1 - delta)) / eval_chebyt(d, 1 / (1 - delta))

d = degree(delta, eps)
x = np.linspace(-(1 - delta), 1 - delta, 10001)
error = np.max(np.abs(filter(x, d)))
test_ok = error <= eps and np.isclose(filter(1, d), 1)

print(f"d={d}, max error={error:.4e}, eps={eps:.4e}: {'PASS' if test_ok else 'FAIL'}")

d=38, max error=9.0642e-03, eps=1.0000e-02: PASS


## Choosing the degree of the spectral filter with uncertain gap estimates

For each step we use $\varepsilon_j=(10^{-2}/4)/L$, where $L$ is the number of schedule temperatures for the instance, and select the even degree
$$
d_c(\widehat\delta_j,\varepsilon_j)=2\left\lceil\frac{c\ln(2/\varepsilon_j)}{2\arccos(1-\widehat\delta_j)}\right\rceil.
$$
We calculate the **reference degree** $d_{\mathrm{actual}}=d_1(\delta_{\mathrm{actual}},\varepsilon_j)$ from the numerical gap and the **predicted degree** $d_c=d_c(\delta_{\mathrm{fit}},\varepsilon_j)$ from the fitted gap, for $c=1,\ldots,6$. Strict coverage requires $d_{\mathrm{actual}}\le d_c$ at **every step** of an instance's schedule within a temperature interval. The reported percentage is the fraction of instances meeting this condition, either for each $n$ separately or pooled across $n=5,\ldots,10$.

In [6]:
def degree(delta, eps, c=1):
    theta = np.arccos(1 - np.asarray(delta))
    return (2 * np.ceil(c * np.log(2 / np.asarray(eps)) / (2 * theta))).astype(int)

C_TESTED = (1, 2, 3, 4, 5, 6)

EPS_TARGET = 1e-2
eps_filter = (EPS_TARGET / 4) / schedule_gaps["L"]
schedule_gaps["d_actual"] = degree(schedule_gaps["gap_actual"], eps_filter)

for c in C_TESTED:
    schedule_gaps[f"d_c{c}"] = degree(schedule_gaps["gap_fit"], eps_filter, c=c)

We first compute **strict coverage**: every filter degree in the interval must be at least the reference degree for that instance.

In [7]:
schedule_gaps["beta_range"] = pd.cut(schedule_gaps["beta"], [0, 1, 2, 4, 16])
columns = [f"d_c{c}" for c in C_TESTED]
keys = ["proposal", "n", "beta_range"]

def summarize_coverage(passed):
    by_instance = passed.groupby(
        [schedule_gaps[k] for k in keys + ["idx"]], observed=True
    ).all()
    result = pd.concat([
        by_instance.groupby(level=keys, observed=True).mean(),
        by_instance.groupby(level=["proposal", "beta_range"], observed=True).mean()
            .assign(n="all").set_index("n", append=True).reorder_levels(keys)
    ]).mul(100)
    result.columns = [f"c={c} (%)" for c in C_TESTED]
    return result

passed_strict = schedule_gaps[columns].ge(schedule_gaps["d_actual"], axis=0)

The resulting strict instance-level coverage percentages are:

In [8]:
coverage_strict = summarize_coverage(passed_strict)
print(coverage_strict.round(2).to_string())

                         c=1 (%)  c=2 (%)  c=3 (%)  c=4 (%)  c=5 (%)  c=6 (%)
proposal n   beta_range                                                      
layden   5   (0, 1]        47.00    99.00    100.0    100.0    100.0    100.0
             (1, 2]        54.00    99.00    100.0    100.0    100.0    100.0
             (2, 4]        54.00   100.00    100.0    100.0    100.0    100.0
             (4, 16]       50.00   100.00    100.0    100.0    100.0    100.0
         6   (0, 1]        42.00   100.00    100.0    100.0    100.0    100.0
             (1, 2]        45.00   100.00    100.0    100.0    100.0    100.0
             (2, 4]        46.00   100.00    100.0    100.0    100.0    100.0
             (4, 16]       43.00   100.00    100.0    100.0    100.0    100.0
         7   (0, 1]        47.00   100.00    100.0    100.0    100.0    100.0
             (1, 2]        51.00   100.00    100.0    100.0    100.0    100.0
             (2, 4]        54.00   100.00    100.0    100.0    1

For the Hamiltonian-simulation proposal, the strict coverage at $c=2$ ranges from $98.67\%$ to $100\%$ after pooling system sizes within each temperature interval; for the uniform proposal it ranges from $99.67\%$ to $100\%$. However, some individual subgroups have lower coverage (for example, $94\%$ for Hamiltonian simulation at $n=10$ and $\beta\in(4,16]$). The choice $c=3$ covers all tested instances under this **reference-degree comparison**, but this is an empirical observation, not a certificate of spectral-filter accuracy on unseen instances.

### Coverage with a 30\% degree tolerance

The strict test counts any predicted degree below the reference degree as a failure. To **allow a 5\% underestimation of the required degree**, we instead accept a step when
$$
d_c\ge\left\lceil0.7\,d_{\mathrm{actual}}\right\rceil,
$$
and again require every step in the temperature interval to pass for an instance to count as covered.

In [16]:
# Genuine tolerance: the predicted degree may fall short of the reference by up to 5%.
passed_tolerant = schedule_gaps[columns].ge(
    np.ceil(0.70 * schedule_gaps["d_actual"]), axis=0
)
coverage_tolerant = summarize_coverage(passed_tolerant)
print("30% underestimation tolerance (per-instance coverage, %):")
print(coverage_tolerant.round(2).to_string())

30% underestimation tolerance (per-instance coverage, %):
                         c=1 (%)  c=2 (%)  c=3 (%)  c=4 (%)  c=5 (%)  c=6 (%)
proposal n   beta_range                                                      
layden   5   (0, 1]        85.00    100.0    100.0    100.0    100.0    100.0
             (1, 2]        91.00    100.0    100.0    100.0    100.0    100.0
             (2, 4]        95.00    100.0    100.0    100.0    100.0    100.0
             (4, 16]       93.00    100.0    100.0    100.0    100.0    100.0
         6   (0, 1]        85.00    100.0    100.0    100.0    100.0    100.0
             (1, 2]        82.00    100.0    100.0    100.0    100.0    100.0
             (2, 4]        86.00    100.0    100.0    100.0    100.0    100.0
             (4, 16]       82.00    100.0    100.0    100.0    100.0    100.0
         7   (0, 1]        93.00    100.0    100.0    100.0    100.0    100.0
             (1, 2]        88.00    100.0    100.0    100.0    100.0    100.0
      